# Exp 17 — Просмотр результатов eval на реальных данных

Запускается после `bash experiments/17/15_eval_real_data.sh`.

Читает два JSON-а:
- `output/07_real_data_test_bge-m3/results.json` — **unsupervised** (category consistency, reciprocal, confidence separation, GNN vs raw baseline)
- `output/bge-m3/v17_views_gat_model_real_trusted_eval.json` — **supervised** (F1/P/R/AUC/AP на gold + silver метках)

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODEL_TAG = "bge-m3"
LOSS = "ntxent"  # "bce" или "ntxent"

suffix = "_bce" if LOSS == "bce" else ""
UNSUP_PATH = Path("../../output") / f"07_real_data_test_{MODEL_TAG}" / "results.json"
SUP_PATH = (Path("../../output") / MODEL_TAG /
            f"v17_views_gat{suffix}_model_real_trusted_eval.json")

print("Unsupervised:", UNSUP_PATH, "→", "OK" if UNSUP_PATH.exists() else "НЕТ")
print("Supervised:  ", SUP_PATH,   "→", "OK" if SUP_PATH.exists() else "НЕТ")

Unsupervised: ../../output/07_real_data_test_bge-m3/results.json → НЕТ
Supervised:   ../../output/bge-m3/v17_views_gat_model_real_trusted_eval.json → НЕТ


## 1. Unsupervised метрики (exp 07)

Категория, reciprocal nearest-neighbor, confidence separation. Сравнение GNN vs raw rubert/bge baseline.

In [ ]:
unsup = json.loads(UNSUP_PATH.read_text())

rows = []
for method in ("GNN", "Baseline"):
    if method not in unsup:
        continue
    r = unsup[method]
    rows.append({
        "method": method,
        "category consistency": r["category_consistency"]["precision"],
        "reciprocal rate":     r["reciprocal"]["rate"],
        "confidence margin":   r.get("confidence", {}).get("mean_margin", None),
    })
df_unsup = pd.DataFrame(rows).set_index("method")
df_unsup.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0)

In [ ]:
metrics = ["category consistency", "reciprocal rate"]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(metrics))
w = 0.35
for i, method in enumerate(df_unsup.index):
    vals = df_unsup.loc[method, metrics].values.astype(float)
    bars = ax.bar(x + (i - 0.5) * w, vals, w, label=method,
                  color="#5b9bd5" if method == "GNN" else "#bbb")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.3f}",
                ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("score")
ax.set_title("Unsupervised: GNN vs Baseline")
ax.legend(); ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, max(1.0, df_unsup[metrics].values.max() * 1.15))
plt.show()

### Per-category breakdown

In [ ]:
per_cat = {}
for method in ("GNN", "Baseline"):
    if method not in unsup:
        continue
    pc = unsup[method]["category_consistency"].get("per_category", {})
    for cat, v in pc.items():
        per_cat.setdefault(cat, {})
        per_cat[cat][f"{method} P"] = v.get("precision", 0)
        per_cat[cat][f"{method} n"] = v.get("total", 0)

df_cat = pd.DataFrame(per_cat).T.sort_values("GNN n", ascending=False)
df_cat

## 2. Supervised метрики (exp 15)

F1/P/R/ROC-AUC/AP на gold (398 пар, ручная разметка) и silver (~26K, LLM-метки).

In [ ]:
sup = json.loads(SUP_PATH.read_text())

rows = []
for tier, r in sup.items():
    for method in ("GNN", "Baseline"):
        if method not in r:
            continue
        m = r[method]
        rows.append({
            "tier":   tier,
            "method": method,
            "n_pairs":   m.get("n_pairs", 0),
            "n_pos":     m.get("n_pos", 0),
            "threshold": m.get("threshold", None),
            "F1":        m.get("f1", None),
            "Precision": m.get("precision", None),
            "Recall":    m.get("recall", None),
            "ROC-AUC":   m.get("roc_auc", None),
            "AP":        m.get("avg_precision", None),
        })
df_sup = pd.DataFrame(rows).set_index(["tier", "method"])
df_sup.style.format({
    "threshold": "{:.3f}", "F1": "{:.3f}", "Precision": "{:.3f}",
    "Recall": "{:.3f}", "ROC-AUC": "{:.3f}", "AP": "{:.3f}",
    "n_pairs": "{:,.0f}", "n_pos": "{:,.0f}",
}, na_rep="—").background_gradient(
    cmap="Greens", subset=["F1", "Precision", "Recall", "ROC-AUC", "AP"], axis=0,
)

In [ ]:
tiers = sorted({t for t, _ in df_sup.index})
metrics_cols = ["F1", "Precision", "Recall", "ROC-AUC", "AP"]

fig, axes = plt.subplots(1, len(tiers), figsize=(5 * len(tiers), 4), squeeze=False)
for j, tier in enumerate(tiers):
    ax = axes[0, j]
    x = np.arange(len(metrics_cols))
    w = 0.35
    for i, method in enumerate(["GNN", "Baseline"]):
        if (tier, method) not in df_sup.index:
            continue
        vals = df_sup.loc[(tier, method), metrics_cols].values.astype(float)
        bars = ax.bar(x + (i - 0.5) * w, vals, w, label=method,
                      color="#5b9bd5" if method == "GNN" else "#bbb")
        for bar, v in zip(bars, vals):
            if v == v:  # not nan
                ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.2f}",
                        ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metrics_cols, rotation=0)
    n_pairs = df_sup.loc[(tier, "GNN"), "n_pairs"] if (tier, "GNN") in df_sup.index else 0
    n_pos = df_sup.loc[(tier, "GNN"), "n_pos"] if (tier, "GNN") in df_sup.index else 0
    ax.set_title(f"{tier}  (n={n_pairs}, +={n_pos})")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

### Threshold sweep — F1 vs порог

In [ ]:
fig, axes = plt.subplots(1, len(tiers), figsize=(5 * len(tiers), 4), squeeze=False)
for j, tier in enumerate(tiers):
    ax = axes[0, j]
    for method, color in [("GNN", "#5b9bd5"), ("Baseline", "#999")]:
        sweep = sup.get(tier, {}).get(method, {}).get("sweep_f1", [])
        if not sweep:
            continue
        sweep = np.asarray(sweep)
        if sweep.ndim == 2:
            ax.plot(sweep[:, 0], sweep[:, 1], label=method, color=color, lw=2)
        else:
            # массив только F1 — генерим x по сетке от 0 до 1
            xs = np.linspace(0, 1, len(sweep))
            ax.plot(xs, sweep, label=method, color=color, lw=2)
        best_thr = sup[tier][method].get("threshold")
        best_f1 = sup[tier][method].get("f1")
        if best_thr is not None and best_f1 is not None:
            ax.axvline(best_thr, color=color, ls="--", alpha=0.4)
            ax.scatter([best_thr], [best_f1], color=color, zorder=5)
    ax.set_xlabel("threshold (cosine sim)")
    ax.set_ylabel("F1")
    ax.set_title(f"{tier}")
    ax.set_ylim(0, 1.05)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Per-strategy breakdown (silver/silver+)

Раскладка по способу формирования меток (например: exact_match / fuzzy / nomatch_high_sim и т.п.).

In [ ]:
strat_rows = []
for tier, r in sup.items():
    for method in ("GNN", "Baseline"):
        per_strat = r.get(method, {}).get("per_strategy", {})
        for strat, m in per_strat.items():
            strat_rows.append({
                "tier": tier, "method": method, "strategy": strat,
                "n": m.get("n", 0), "n_pos": m.get("n_pos", 0),
                "F1": m.get("f1"), "P": m.get("precision"), "R": m.get("recall"),
            })

if strat_rows:
    df_strat = pd.DataFrame(strat_rows)
    pivot = df_strat.pivot_table(
        index=["tier", "strategy", "n", "n_pos"],
        columns="method", values="F1",
    ).reset_index()
    pivot = pivot.sort_values(["tier", "n"], ascending=[True, False])
    display(pivot.style.format({"GNN": "{:.3f}", "Baseline": "{:.3f}"}, na_rep="—"))
else:
    print("per_strategy не найден — поле было пустым")

## 3. Сводка одной таблицей

In [ ]:
print(f"=== Real-data eval: bge-m3 / v17_views / {LOSS} ===\n")
print("Unsupervised:")
for method in df_unsup.index:
    cc = df_unsup.loc[method, "category consistency"]
    rr = df_unsup.loc[method, "reciprocal rate"]
    print(f"  {method:9s}  category={cc:.3f}   reciprocal={rr:.3f}")

print("\nSupervised:")
for (tier, method), row in df_sup.iterrows():
    print(f"  {tier:8s} / {method:9s}  F1={row['F1']:.3f}  "
          f"P={row['Precision']:.3f}  R={row['Recall']:.3f}  "
          f"AUC={row['ROC-AUC']:.3f}  AP={row['AP']:.3f}")